# PC-FNO3D NATIONAL v4 — ATOM Wind en ligne (atom.zelqane.com)

Sert le **modèle national livré** (C' graine 44, `decision_v4_national.json` : six DSO conformes sur le test 2022, −21 % d'erreur face à l'ERA5 de 0 à 1 km) à l'application ATOM Wind, avec la **couche de vent à 10 m** qui intègre les mesures des 12 aéroports (`pcfno3d.vent10m_oi`, règle `regle_produit_vent10m_v4.json`, `f07973e1bd73…` : −3,7 % face à l'ERA5 sur 2023–2024).

Même adresse que le Grand Tunis (tunnel nommé `atom`) : pendant que ce carnet tourne, atom.zelqane.com sert la Tunisie entière. Instants servis : les 180 instants du test 2022 (cache) et le temps réel (GFS + METAR).

**Avant** : secret Colab `TUNNEL_CRED_CONTENTS` (remis par Rayen), avec « Accès au notebook ». **GPU** : L4 demandé (inférence seulement, ~2 unités/h) ; en cas de manque de mémoire : A100.


## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")   # code et poids du Grand Tunis
NAT = Path("/content/drive/MyDrive/pcfno3d_national")
NAT.mkdir(parents=True, exist_ok=True)
JOURNAUX = NAT / "journaux"
JOURNAUX.mkdir(exist_ok=True)
libre = shutil.disk_usage("/content/drive/MyDrive").free / 1e9
print(f"place libre sur le Drive : {libre:.0f} Go (le jeu national en demande ~50)")

CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille nationale, grille_reelle.json, forcage ERA5 (une archive sur le Drive)
DON = Path("/content/donnees")
G = DON / "national"
if not (G / "forcage_era5.csv").exists():
    with zipfile.ZipFile(NAT / "national_donnees.zip") as zf:
        zf.extractall(DON)
print(sorted(p.name for p in G.iterdir()))

# code du depot public au commit fige (remplace l'archive du Drive, plus ancienne)
COMMIT = "ff86ed81ad3dc4c04aa9d7bce5d4203052fc02e3"
os.chdir("/content")    # le noyau etait dans /content/PFE : le supprimer laisserait pip dans un dossier efface
subprocess.run(f"rm -rf /content/PFE && git init -q /content/PFE && cd /content/PFE && git remote add origin "
               f"https://github.com/ines-rouatbi/pcfno3d.git && git fetch -q --depth 1 origin {COMMIT} "
               f"&& git checkout -q FETCH_HEAD", shell=True, check=True)
os.chdir("/content/PFE")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "/content/PFE"],
                   capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:], r.stderr[-3000:])
    raise SystemExit("pip install -e /content/PFE a echoue (sortie ci-dessus)")
import importlib
importlib.invalidate_caches()
import hashlib
for f, h in (("regle_produit_vent10m_v4.json", "f07973e1bd7329492d87fb8e043d4cf1be4148b3e041639199332fb834cc1104"), ("regle_choix_v4_Cp_s44.json", "19cee4e50461e75a74f1ad54db7c4fb9496c3f5be112e65bcbb43278862ce4a5"), ("regle_w_servi_v4.json", "4e5453455d67c2e0b397adbe88852d47f898c6827fa250a933f588579460e5b3"), ("regle_v4_national.json", "60671e843ee648ee2422168a7659a53dfe8aa414be7b1c7c6f54eeec622a43f8"),):
    assert hashlib.sha256((CODE / "configs" / "national" / f).read_bytes()).hexdigest() == h, f
print("code", COMMIT, "| regle v4 verifiee")

## 2. Cache v3 (instants du test 2022)

In [ ]:
LOC = Path("/content/national")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (NAT / f"cache_v3_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_v3_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(NAT / f"cache_v3_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(NAT / "jeu_v3.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())
# controle de la verite deja fait par les carnets d'entrainement

## 3. Configuration de service (modèle national + couche 10 m)

In [ ]:
import yaml
RUN = NAT / "runs" / "national_1km_v4_Cp_s44"      # modele national livre (decision_v4_national.json)
assert (RUN / "best.pt").exists(), RUN
c = yaml.safe_load(open(CODE / "configs" / "national_1km_colab.yaml"))   # w_continuite + w_beta 0,75
c["name"] = "national_v4_service"
c["paths"]["run_dir"], c["paths"]["local_dir"] = str(RUN), "/content/run_local_national_service"
c["paths"]["stations_obs"] = str(G / "obs")       # vraies mesures METAR des 12 aeroports
c.setdefault("inference", {})["couche_10m_oi"] = True
yaml.safe_dump(c, open(CODE / "configs" / "national_v4_service.yaml", "w"), sort_keys=False)
print("config national_v4_service :", c["inference"])
!pip install -q fastapi uvicorn python-multipart httpx mcp==2.3.0 anthropic 2>&1 | tail -2


## 4. Démarrer le service derrière le tunnel nommé

In [ ]:
from google.colab import userdata
from service.colab import serve_public

TUNNEL_ID = "e552f718-2478-4fe3-82b4-8d33a4b9eb6e"
HOTE = "atom-origin.zelqane.com"
CLE_NOM = "atom-worker"                      # la cle de lecture du Worker de Rayen (empreinte seulement)
CLE_EMPREINTE = "ce7965c54284eb6d1a274119839a71d9785731609d04b17bd08fb6da7d2c30fd"
try:
    identifiants = userdata.get("TUNNEL_CRED_CONTENTS")
except Exception:
    raise RuntimeError("secret Colab TUNNEL_CRED_CONTENTS introuvable ou non autorise : panneau de gauche, icone cle, "
                       "ajoute-le (contenu remis par Rayen) et active « Acces au notebook ».") from None
try:
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("assistant : cle presente")
except Exception:
    print("assistant : pas de secret ANTHROPIC_API_KEY (assistant desactive)")
info = serve_public(config="national_v4_service", key_name=CLE_NOM, scope="read", key_hash=CLE_EMPREINTE,
                    quota_minute=1200, named_tunnel={"id": TUNNEL_ID, "hostname": HOTE, "credentials": identifiants})
del identifiants


## 5. Vérifier de l'extérieur

In [ ]:
import urllib.request
req = urllib.request.Request(f"https://{HOTE}/api/health", headers={"User-Agent": "pcfno3d-colab/1.0"})
with urllib.request.urlopen(req, timeout=120) as r:
    sante = json.loads(r.read().decode())
print(json.dumps(sante, ensure_ascii=False, indent=1))
assert sante["status"] == "ok", "le service ne repond pas correctement par le tunnel"
print(f"\nEN LIGNE : https://{HOTE}/app/atom-wind/ — modele NATIONAL (C' s44), couche 10 m avec les aeroports.")


## 6. Garder en vie

In [ ]:
try:
    while True:
        tunnel_ok = info["tunnel"].alive
        service_ok = info["process"].poll() is None
        print(time.strftime("%H:%M"), "| tunnel", "ok" if tunnel_ok else "ARRETE",
              "| service", "ok" if service_ok else "ARRETE", flush=True)
        if not (tunnel_ok and service_ok):
            print("journaux :", info["log_path"], "et", info["tunnel"].log_path)
            break
        time.sleep(600)
except KeyboardInterrupt:
    print("surveillance arretee — le service tourne toujours")


In [ ]:
# Garde-fou : « Tout executer » atteint aussi cette cellule des que la surveillance s'arrete.
ARRETER = False
if ARRETER:
    from service.colab import stop
    stop(info)
    from google.colab import runtime
    runtime.unassign()
else:
    print("service laisse en marche (ARRETER = False)")
